# Student Dropout Prediction (EDA + Streamlit)

This notebook walks through the full project: data exploration, model training, and building a Streamlit dashboard.

In [ ]:
# Setup and imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

%matplotlib inline

## 1) Setup Streamlit Project Structure

We keep a clean project structure with the following folders:

- `data/` — raw and processed datasets
- `src/` — supporting helper modules and data processing functions
- `app/` — Streamlit app files and dashboard components
- `notebooks/` — exploratory analysis and development notebooks

The code in this notebook will read from `data/student_dropout_dataset.csv` and save models to the project root.

## 2) Install Dependencies

Run this in your activated virtual environment (PowerShell example):

```bash
pip install -r requirements.txt
```

The `requirements.txt` contains:

- pandas
- matplotlib
- seaborn
- scikit-learn
- joblib
- streamlit


## 3) Load and Inspect Data

Load the dataset and perform a quick sanity check (head, info, missing values).

In [ ]:
DATA_PATH = "../data/student_dropout_dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Basic dataset checks
print(df.info())
print("\nMissing values per column:\n", df.isnull().sum())

# Drop any missing values (if found)
df = df.dropna().reset_index(drop=True)


## 4) Create Visualizations in Notebook

Explore the relationships between features and `dropout`.


In [ ]:
# Dropout distribution
plt.figure(figsize=(8, 5))
sns.countplot(x="dropout", data=df)
plt.title("Dropout Distribution")
plt.show()

# Attendance vs Dropout
plt.figure(figsize=(8, 5))
sns.boxplot(x="dropout", y="attendance_percentage", data=df)
plt.title("Attendance vs Dropout")
plt.show()

# Stress Level vs Dropout
plt.figure(figsize=(8, 5))
sns.boxplot(x="dropout", y="stress_level", data=df)
plt.title("Stress Level vs Dropout")
plt.show()

# Study hours vs grade
plt.figure(figsize=(8, 5))
sns.scatterplot(x="study_hours_per_week", y="previous_grade", hue="dropout", data=df)
plt.title("Study Hours vs Previous Grade")
plt.show()

# Correlation heatmap
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

## 5) Export Plots for Streamlit

In a Streamlit app, it’s often useful to save plots to disk or wrap them in functions that can be reused.


In [ ]:
import os

PLOT_DIR = "../app/assets"
os.makedirs(PLOT_DIR, exist_ok=True)


def save_figure(fig, filename: str):
    path = os.path.join(PLOT_DIR, filename)
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved plot to {path}")
    return path


# Example: save the dropout distribution plot
fig = plt.figure(figsize=(8, 5))
sns.countplot(x="dropout", data=df)
plt.title("Dropout Distribution")
fig_path = save_figure(fig, "dropout_distribution.png")
plt.close(fig)


## 6) Build Streamlit App Skeleton

We created a Streamlit app in `app/app.py` that loads the pre-trained model and exposes an interactive interface. You can run it locally with:

```bash
streamlit run app/app.py
```

This app uses the same model artifacts (`model.pkl` and `label_encoder.pkl`) generated by `model_training.py`.


## 7) Add Utility and Config Files

This project uses:

- `requirements.txt` to lock dependencies
- `.gitignore` to exclude virtual environment and artifacts
- `config.json` to store paths to dataset and model files
- `src/data_utils.py` for reusable data loading and preprocessing functions

You can modify `config.json` to point to other dataset locations or model file names.


## 8) Run and Test Streamlit Locally

From the project root, run:

```bash
streamlit run app/app.py
```

Then open the URL shown in the terminal (usually `http://localhost:8501`).

If you make changes to the app, Streamlit will auto-reload.


## 9) Document Project Files and Folder Layout

**Project structure**

```
VIBE CODING/
├─ app/
│  └─ app.py              # Streamlit application entrypoint
├─ data/
│  └─ student_dropout_dataset.csv
├─ notebooks/
│  └─ streamlit_eda_model.ipynb  # This notebook
├─ src/
│  └─ data_utils.py       # Helper functions for loading/preprocessing data
├─ model_training.py      # Train models and save artifacts
├─ eda.py                # Quick explorations and plots
├─ requirements.txt      # Dependencies for pip install
├─ config.json           # Configurable paths for dataset/model
├─ README.md             # Project overview and quick start
├─ .gitignore            # Files/folders to ignore in git
```

Each script is designed to be runnable from the project root, making it easy to follow the end-to-end pipeline from data to deployment.

## 6) Model Training + Confusion Matrix

This section trains a Random Forest model on the dataset and displays the confusion matrix to assess performance.


In [ ]:
# Train a Random Forest classifier and show confusion matrix

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Prepare data
le = LabelEncoder()
df["gender"] = le.fit_transform(df["gender"])

X = df.drop("dropout", axis=1)
y = df["dropout"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print("\nClassification Report:\n", classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## Extra: Histograms for All Numeric Features

Plot distributions of all numeric columns in the dataset for quick overview.


In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

fig, axes = plt.subplots(len(numeric_cols)//3 + 1, 3, figsize=(16, 4*(len(numeric_cols)//3 + 1)))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], bins=20, kde=True, ax=ax)
    ax.set_title(col)

# Turn off unused subplots
for ax in axes[len(numeric_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()
